<a href="https://colab.research.google.com/github/victorshi119/LING-L-690JoparaASRModel/blob/LING-L-690JOPARAASRMODEL/OfficialModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Change to your working directory
%cd /content/drive/MyDrive/690IndependentStudy

/content/drive/MyDrive/690IndependentStudy


# Data Preparation

In [4]:
import pandas as pd

# Load the CSV with utterance metadata
csv_file = "tr-005.csv"  # CSV containing columns like 'Tiempo Inicio', 'Tiempo Fin', 'Contenido', etc.
data_df = pd.read_csv(csv_file)

# Structure to hold our dataset entries
utterances = []
for idx, row in data_df.iterrows():
    start_time = float(row['Tiempo Inicio'])
    end_time = float(row['Tiempo Fin'])
    text = str(row['Contenido'])
    # If there's an utterance ID column (like '#' or index), use it; otherwise use the row index
    utt_id = row.get('#', idx+1) if hasattr(row, 'get') else (row['#'] if '#' in row else idx+1)
    utterances.append({
        "id": int(utt_id),
        "start": start_time,
        "end": end_time,
        "text": text
    })

print(f"Loaded {len(utterances)} utterances from CSV.")
print("Example utterance:", utterances[0])


Loaded 1699 utterances from CSV.
Example utterance: {'id': 1, 'start': 0.11, 'end': 0.78, 'text': 'ahí ya puse'}


In [9]:
import re

def parse_tokenized_tagged(filepath):
    utterances = {}
    cur_id = None
    tokens = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('```'):
                continue
            if line.startswith('#'):
                if cur_id is not None and tokens:
                    utterances[cur_id] = tokens
                # Extract utterance id, e.g. #005#3 . .
                m = re.match(r'#(\d+)#(\d+)', line)
                if m:
                    file_id = m.group(1)
                    utt_num = m.group(2)
                    cur_id = f"{file_id}-{utt_num}"
                    tokens = []
            else:
                cols = line.split('\t')
                if len(cols) == 3:
                    _, token, tag = cols
                    tokens.append((token, tag))
        if cur_id is not None and tokens:
            utterances[cur_id] = tokens
    return utterances

# Usage:
utterances = parse_tokenized_tagged("tr-005-tokenized_tagged.txt")
print(utterances["005-3"])
# Output: [('podés', 's'), ('contarnos', 's'), ('tu', 's'), ('nombre', 's'), ('?', 'o')]


[('podés', 's'), ('contarnos', 's'), ('tu', 's'), ('nombre', 's'), ('?', 'o')]


In [13]:
import pandas as pd

csv_file = "tr-005.csv"
data_df = pd.read_csv(csv_file)

utterance_meta = {}
file_num = "005"  # You may want to generalize this, for now hardcoded

for idx, row in data_df.iterrows():
    # Use the '#' column as the utterance number (if present)
    if '#' in row:
        utt_num = str(row['#']).strip()
    else:
        utt_num = str(idx+1)
    utt_id = f"{file_num}-{utt_num}"
    utterance_meta[utt_id] = {
        "start": float(row['Tiempo Inicio']),
        "end": float(row['Tiempo Fin']),
        "text": str(row['Contenido'])
    }

print(f"Loaded {len(utterance_meta)} utterances from CSV.")
print("Example utterance:", list(utterance_meta.items())[0])


Loaded 1699 utterances from CSV.
Example utterance: ('005-1', {'start': 0.11, 'end': 0.78, 'text': 'ahí ya puse'})


# Data Prep pt2

In [4]:
import re

def parse_tokenized_tagged(filepath):
    utterances = {}
    cur_id = None
    tokens = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('```'):
                continue
            if line.startswith('#'):
                if cur_id is not None and tokens:
                    utterances[cur_id] = tokens
                # Extract utterance id, e.g. #005#3 . .
                m = re.match(r'#(\d+)#(\d+)', line)
                if m:
                    file_id = m.group(1)
                    utt_num = m.group(2)
                    cur_id = f"{file_id}-{utt_num}"
                    tokens = []
            else:
                cols = line.split('\t')
                if len(cols) == 3:
                    _, token, tag = cols
                    tokens.append((token, tag))
        if cur_id is not None and tokens:
            utterances[cur_id] = tokens
    return utterances

# Example usage:
token_file = "tr-005-tokenized_tagged.txt"
tokenized_utterances = parse_tokenized_tagged(token_file)
print(list(tokenized_utterances.items())[:3])


[('005-1', [('ahí', 's'), ('ya', 's'), ('puse', 's')]), ('005-2', [('ahí', 's'), ('ya', 's'), ('está', 's')]), ('005-3', [('podés', 's'), ('contarnos', 's'), ('tu', 's'), ('nombre', 's'), ('?', 'o')])]


In [5]:
import pandas as pd

csv_file = "tr-005.csv"
data_df = pd.read_csv(csv_file)
file_num = "005"  # Set this for each file

utterance_meta = {}
for idx, row in data_df.iterrows():
    utt_num = str(row['#']).strip() if '#' in row else str(idx+1)
    utt_id = f"{file_num}-{utt_num}"
    utterance_meta[utt_id] = {
        "start": float(row['Tiempo Inicio']),
        "end": float(row['Tiempo Fin']),
        "text": str(row['Contenido'])
    }

print(f"Loaded {len(utterance_meta)} utterances from CSV.")
print(list(utterance_meta.items())[:3])


Loaded 1699 utterances from CSV.
[('005-1', {'start': 0.11, 'end': 0.78, 'text': 'ahí ya puse'}), ('005-2', {'start': 3.08, 'end': 3.81, 'text': 'ahí ya está'}), ('005-3', {'start': 7.12, 'end': 9.24, 'text': 'podés contarnos tu nombre?'})]


# Pseudo Frame-Level Label Generation (Language Alignment)
Using the start and end times, we can extract the audio for each utterance and determine how many frames the Wav2Vec2 encoder will produce for that segment. For 16 kHz audio, the Wav2Vec2 encoder produces roughly 50 feature frames per second (~20 ms apart)
proceedings.neurips.cc
. We will use this to approximate the number of frames for each utterance, then distribute those frames across the transcript’s tokens in order. For each token in an utterance, frames corresponding to that token’s portion of time will be labeled with the token’s language. Since we don’t have exact word-level timestamps, we assume an even split of the utterance’s frames among the tokens (this is a simplification for demonstration). Frames corresponding to tokens labeled 's' get label 0 (Spanish), 'g' get label 1 (Guaraní), and 'o' (punctuation or silence) frames will be marked with -100 so that they are ignored in the loss computation.

In [6]:
import math
import numpy as np
import soundfile as sf
import librosa
import re

audio_file = "tr-005.wav"
audio_sf = sf.SoundFile(audio_file)
original_sr = audio_sf.samplerate  # original sample rate of the audio file
print(f"Audio sample rate: {original_sr} Hz")

dataset = []

for utt_id in utterance_meta:
    # Check if you have token-level tags for this utterance
    if utt_id not in tokenized_utterances:
        continue
    meta = utterance_meta[utt_id]
    tokens = tokenized_utterances[utt_id]

    start, end, text = meta["start"], meta["end"], meta["text"]
    duration = end - start
    if duration <= 0 or not tokens:
        continue

    # Audio
    start_frame = int(start * original_sr)
    num_frames = int(duration * original_sr)
    audio_sf.seek(start_frame)
    audio_segment = audio_sf.read(frames=num_frames, dtype='float32')
    # If stereo, convert to mono
    if audio_segment.ndim > 1:
        audio_segment = audio_segment.mean(axis=1)
    # Resample to 16kHz for Wav2Vec2
    target_sr = 16000
    if original_sr != target_sr:
        audio_segment = librosa.resample(audio_segment, orig_sr=original_sr, target_sr=target_sr)
    # Estimate encoder frames (approx one per 20ms of audio)
    frame_count = math.ceil(len(audio_segment) / 320)
    M = len(tokens)
    if M == 0 or frame_count == 0:
        continue

    # Generate frame-level language labels
    frame_labels = []
    for f in range(frame_count):
        token_index = min(int(f * M / frame_count), M-1)
        _, lang_tag = tokens[token_index]
        if lang_tag == 's':
            frame_labels.append(0)      # Spanish
        elif lang_tag == 'g':
            frame_labels.append(1)      # Guarani
        else:
            frame_labels.append(-100)   # Ignore in loss
    frame_labels = np.array(frame_labels, dtype=np.int64)
    # Clean transcript for ASR
    clean_text = re.sub(r'[.,?!¿¡()]', '', text).strip().lower()

    dataset.append({
        "utt_id": utt_id,
        "audio": audio_segment,
        "text": clean_text,
        "lid_labels": frame_labels
    })

audio_sf.close()
print(f"Prepared {len(dataset)} training examples with frame-level language labels.")
print("Example:", dataset[0]['utt_id'], dataset[0]['text'], dataset[0]['lid_labels'][:20])


Audio sample rate: 44100 Hz
Prepared 1679 training examples with frame-level language labels.
Example: 005-1 ahí ya puse [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [7]:
import pickle

# Save as a pickle for easy PyTorch loading
with open("tr-005-prepared.pkl", "wb") as f:
    pickle.dump(dataset, f)
print("Saved processed dataset to tr-005-prepared.pkl")


Saved processed dataset to tr-005-prepared.pkl


#  Build Vocab & Model


In [7]:
all_text = " ".join(item["text"] for item in dataset)
chars = sorted(set(all_text))
if " " not in chars:
    chars.append(" ")
vocab_list = ["<pad>", "<blank>"] + chars
vocab_size = len(vocab_list)
print("Vocab size:", vocab_size)


Vocab size: 50


In [8]:
from transformers import Wav2Vec2Model, Wav2Vec2Config
import torch
import torch.nn as nn
from transformers.modeling_outputs import CausalLMOutput

base_model_name = "facebook/wav2vec2-xls-r-300m"  # Multilingual is safest!
config = Wav2Vec2Config.from_pretrained(base_model_name)
config.vocab_size = vocab_size
config.pad_token_id = 0
config.ctc_loss_reduction = "mean"
config.ctc_zero_infinity = True

class Wav2Vec2ForCSASR(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(base_model_name, config=cfg)
        self.ctc_head = nn.Linear(cfg.hidden_size, cfg.vocab_size)
        self.lid_head = nn.Linear(cfg.hidden_size, 2)
        nn.init.normal_(self.ctc_head.weight, mean=0.0, std=0.1)
        nn.init.normal_(self.lid_head.weight, mean=0.0, std=0.1)
        nn.init.zeros_(self.ctc_head.bias)
        nn.init.zeros_(self.lid_head.bias)
        self.ctc_loss_fn = nn.CTCLoss(blank=1, zero_infinity=True)
    def forward(self, input_values, attention_mask=None, labels=None, lid_labels=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        logits = self.ctc_head(hidden_states)
        lid_logits = self.lid_head(hidden_states)
        log_probs = nn.functional.log_softmax(logits, dim=-1)
        loss = None
        if labels is not None:
            log_probs_t = log_probs.permute(1, 0, 2)
            batch_size, max_frames, _ = logits.shape
            input_lengths = attention_mask.sum(dim=1) if attention_mask is not None else torch.full((batch_size,), max_frames, dtype=torch.long)
            target_list, target_lengths = [], []
            for seq in labels:
                mask = seq != -100
                target_list.append(seq[mask])
                target_lengths.append(mask.sum().item())
            target_lengths = torch.tensor(target_lengths, dtype=torch.long)
            targets_concat = torch.cat(target_list) if target_list else torch.tensor([], dtype=torch.long)
            ctc_loss = self.ctc_loss_fn(log_probs_t, targets_concat, input_lengths, target_lengths)
            lid_loss = 0.0
            if lid_labels is not None:
                B, T, _ = lid_logits.shape
                lid_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
                lid_loss = lid_loss_fn(lid_logits.view(B * T, 2), lid_labels.view(-1))
            loss = ctc_loss + lid_loss
        return CausalLMOutput(loss=loss, logits=logits)

model = Wav2Vec2ForCSASR(config)
print("Model initialized.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Model initialized.


# Data Loader and Trainer

In [9]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
import torch
import math

# === 1. Check your data before proceeding ===
print("Checking dataset type and first item:")
print(type(dataset))           # Should be <class 'list'>
print(len(dataset))            # Should be a positive integer
print(dataset[0].keys())       # Should have utt_id, audio, text, lid_labels
print(dataset[0])              # Preview a sample

# === 2. Custom Dataset Class ===
class CSDataset(Dataset):
    def __init__(self, data_list):
        assert isinstance(data_list, list) and isinstance(data_list[0], dict), "data_list must be a list of dicts"
        self.data = data_list
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = CSDataset(dataset)

# === 3. Your vocab_list must be available ===
# Example: vocab_list = ["<pad>", "<blank>", ... all unique chars ...]
print(f"Vocab size: {len(vocab_list)}. First 10 tokens: {vocab_list[:10]}")

# === 4. Custom collate function ===
def collate_batch(batch):
    print(f"\ncollate_batch called with batch of size: {len(batch)}")
    for i, item in enumerate(batch):
        print(f"Item {i} keys: {list(item.keys())}")
        print(f"utt_id: {item.get('utt_id')}")
        print(f"text: {item.get('text', '<MISSING>')}")
    max_audio_len = max(len(item["audio"]) for item in batch)
    max_frame_count = math.ceil(max_audio_len / 320)
    max_label_len = max(len(item["text"]) for item in batch)
    input_values_list, attention_mask_list, labels_list, lid_labels_list = [], [], [], []
    for item in batch:
        # Audio
        audio = item["audio"]
        aud_tensor = torch.from_numpy(audio.astype('float32'))
        if len(aud_tensor) < max_audio_len:
            pad_len = max_audio_len - len(aud_tensor)
            aud_tensor = torch.nn.functional.pad(aud_tensor, (0, pad_len))
        input_values_list.append(aud_tensor)
        # Attention mask
        attn_mask = torch.ones(len(audio), dtype=torch.long)
        if len(attn_mask) < max_audio_len:
            pad_len = max_audio_len - len(attn_mask)
            attn_mask = torch.nn.functional.pad(attn_mask, (0, pad_len), value=0)
        attention_mask_list.append(attn_mask)
        # Transcription labels
        text = item["text"]
        char_ids = [vocab_list.index(ch) for ch in text if ch in vocab_list]
        label_ids = torch.tensor(char_ids, dtype=torch.long)
        if len(label_ids) < max_label_len:
            pad_size = max_label_len - len(label_ids)
            label_ids = torch.nn.functional.pad(label_ids, (0, pad_size), value=-100)
        labels_list.append(label_ids)
        # LID labels
        lid_labels = torch.tensor(item["lid_labels"], dtype=torch.long)
        frame_count = math.ceil(len(audio) / 320)
        if frame_count < max_frame_count:
            pad_frames = max_frame_count - frame_count
            lid_labels = torch.nn.functional.pad(lid_labels, (0, pad_frames), value=-100)
        lid_labels_list.append(lid_labels)
    input_values = torch.stack(input_values_list)
    attention_mask = torch.stack(attention_mask_list)
    labels = torch.stack(labels_list)
    lid_labels = torch.stack(lid_labels_list)
    print("Batch loaded!")  # Debug print
    return {
        "input_values": input_values,
        "attention_mask": attention_mask,
        "labels": labels,
        "lid_labels": lid_labels
    }

# === 5. Training arguments ===
training_args = TrainingArguments(
    output_dir="./csasr-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=1e-4,
    gradient_accumulation_steps=2,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    report_to="none"
)

# === 6. Trainer ===
trainer = Trainer(
    model=model,   # <-- Your model must be defined (Wav2Vec2 with LAL)
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_batch
)

# === 7. Start Training (Uncomment to run) ===
# trainer.train()
# trainer.save_model("./csasr_model_with_LAL")


Checking dataset type and first item:
<class 'list'>
1679
dict_keys(['utt_id', 'audio', 'text', 'lid_labels'])
{'utt_id': '005-1', 'audio': array([-0.00198044, -0.00770624, -0.00962613, ..., -0.00369842,
       -0.00337931, -0.00352897], dtype=float32), 'text': 'ahí ya puse', 'lid_labels': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}
Vocab size: 50. First 10 tokens: ['<pad>', '<blank>', ' ', "'", '-', 'a', 'b', 'c', 'd', 'e']


In [10]:
from torch.utils.data import DataLoader

loader = DataLoader(train_dataset, batch_size=8, collate_fn=collate_batch)

# Fetch a batch (will print batch info)
for batch in loader:
    print("Batch keys:", batch.keys())
    print("input_values shape:", batch["input_values"].shape)
    break



collate_batch called with batch of size: 8
Item 0 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-1
text: ahí ya puse
Item 1 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-2
text: ahí ya está
Item 2 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-3
text: podés contarnos tu nombre
Item 3 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-4
text: mi nombre es mg
Item 4 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-5
text: cuántos años pa tenés
Item 5 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-6
text: cincuenta y cinco
Item 6 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-7
text: cincuenta y cinco años ña m se queda todo dura ya inaudible
Item 7 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-8
text: jaja
Batch loaded!
Batch keys: dict_keys(['input_values', 'attention_mask', 'labels', 'lid_labels'])
input_values shape: torch.Size([8, 51040])


In [13]:
# Double check vocab list
# Build character vocab from all your training text
all_text = " ".join(item["text"] for item in dataset)
chars = sorted(set(all_text))
if " " not in chars: chars.append(" ")
vocab_list = ["<pad>", "<blank>"] + chars
print(f"Vocab size: {len(vocab_list)}. First 10 tokens:", vocab_list[:10])


Vocab size: 50. First 10 tokens: ['<pad>', '<blank>', ' ', "'", '-', 'a', 'b', 'c', 'd', 'e']


In [15]:
# Inspect label IDs and their mapping
print("Text:", dataset[0]["text"])
print("Label IDs:", [vocab_list.index(ch) for ch in dataset[0]["text"] if ch in vocab_list])
print("Corresponding tokens:", [vocab_list[vocab_list.index(ch)] for ch in dataset[0]["text"] if ch in vocab_list])


Text: ahí ya puse
Label IDs: [5, 12, 35, 2, 29, 5, 2, 20, 25, 23, 9]
Corresponding tokens: ['a', 'h', 'í', ' ', 'y', 'a', ' ', 'p', 'u', 's', 'e']


In [18]:
# This should be your big list with the full dicts, not just lid_labels!
print(type(dataset))
print(len(dataset))
print(dataset[0].keys())
print(dataset[0])

train_dataset = CSDataset(dataset)


<class 'list'>
1679
dict_keys(['utt_id', 'audio', 'text', 'lid_labels'])
{'utt_id': '005-1', 'audio': array([-0.00198044, -0.00770624, -0.00962613, ..., -0.00369842,
       -0.00337931, -0.00352897], dtype=float32), 'text': 'ahí ya puse', 'lid_labels': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}


In [36]:
trainer.train()
trainer.save_model("./csasr_model_with_LAL")



collate_batch called with batch of size: 8
Item 0 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 1 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 2 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 3 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 4 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 5 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 6 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 7 keys: ['lid_labels']
utt_id: None
text: <MISSING>


KeyError: 'audio'

In [23]:
# Manually check!
sample_batch = [train_dataset[i] for i in range(8)]
out = collate_batch(sample_batch)
print("Batch keys:", out.keys())
print("input_values shape:", out["input_values"].shape)



collate_batch called with batch of size: 8
Item 0 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-1
text: ahí ya puse
Item 1 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-2
text: ahí ya está
Item 2 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-3
text: podés contarnos tu nombre
Item 3 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-4
text: mi nombre es mg
Item 4 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-5
text: cuántos años pa tenés
Item 5 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-6
text: cincuenta y cinco
Item 6 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-7
text: cincuenta y cinco años ña m se queda todo dura ya inaudible
Item 7 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-8
text: jaja
Batch loaded!
Batch keys: dict_keys(['input_values', 'attention_mask', 'labels', 'lid_labels'])
input_values shape: torch.Size([8, 51040])


In [30]:
print (collate_batch)

<function collate_batch at 0x7eafd08c49a0>


In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_batch
)
trainer.train()



collate_batch called with batch of size: 8
Item 0 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 1 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 2 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 3 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 4 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 5 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 6 keys: ['lid_labels']
utt_id: None
text: <MISSING>
Item 7 keys: ['lid_labels']
utt_id: None
text: <MISSING>


KeyError: 'audio'

# Old Code Checking Errors

In [25]:
print(type(dataset))
print(type(dataset[0]))
print(dataset[0].keys())
print(dataset[0])


<class 'list'>
<class 'dict'>
dict_keys(['utt_id', 'audio', 'text', 'lid_labels'])
{'utt_id': '005-1', 'audio': array([-0.00198044, -0.00770624, -0.00962613, ..., -0.00369842,
       -0.00337931, -0.00352897], dtype=float32), 'text': 'ahí ya puse', 'lid_labels': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}


In [27]:
for i, ex in enumerate(dataset):
    if not isinstance(ex, dict):
        !print(f"Entry {i} is not a dict: {type(ex)}")
    elif "audio" not in ex:
        !print(f"Entry {i} is missing 'audio': {ex.keys()}")
    elif ex["audio"] is None:
        !print(f"Entry {i} has None for 'audio'")
    elif len(ex["audio"]) == 0:
        !print(f"Entry {i} has empty audio")


In [31]:
print(f"Train dataset length: {len(train_dataset)}")
print(train_dataset[0])



Train dataset length: 1679
{'utt_id': '005-1', 'audio': array([-0.00198044, -0.00770624, -0.00962613, ..., -0.00369842,
       -0.00337931, -0.00352897], dtype=float32), 'text': 'ahí ya puse', 'lid_labels': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}


In [32]:
from torch.utils.data import DataLoader

def collate_batch(batch):
    print("collate_batch called with batch of size:", len(batch))
    for item in batch:
        print(item["utt_id"], item["text"])
    # Minimal dummy return just for debug!
    return batch

loader = DataLoader(train_dataset, batch_size=2, collate_fn=collate_batch)
for batch in loader:
    print("Batch loaded!")
    break


collate_batch called with batch of size: 2
005-1 ahí ya puse
005-2 ahí ya está
Batch loaded!


In [33]:
def collate_batch(batch):
    print("Collate batch debug: batch size", len(batch))
    # Find max lengths
    max_audio_len = max(len(item["audio"]) for item in batch)
    max_label_len = max(len(item["lid_labels"]) for item in batch)
    input_values, attention_mask, lid_labels = [], [], []
    for item in batch:
        audio = item["audio"]
        aud_tensor = torch.from_numpy(audio.astype('float32'))
        # Pad audio
        if len(aud_tensor) < max_audio_len:
            aud_tensor = torch.nn.functional.pad(aud_tensor, (0, max_audio_len - len(aud_tensor)))
        input_values.append(aud_tensor)
        mask = torch.ones(len(audio), dtype=torch.long)
        if len(mask) < max_audio_len:
            mask = torch.nn.functional.pad(mask, (0, max_audio_len - len(mask)), value=0)
        attention_mask.append(mask)
        # Pad lid labels
        lid = torch.tensor(item["lid_labels"], dtype=torch.long)
        if len(lid) < max_label_len:
            lid = torch.nn.functional.pad(lid, (0, max_label_len - len(lid)), value=-100)
        lid_labels.append(lid)
    input_values = torch.stack(input_values)
    attention_mask = torch.stack(attention_mask)
    lid_labels = torch.stack(lid_labels)
    # Add a dummy "labels" key as required (you will update this for CTC or text labels)
    labels = lid_labels.clone()  # Placeholder, replace with proper text label tensor for ASR training
    print("Shapes -- input_values:", input_values.shape, "lid_labels:", lid_labels.shape)
    return {"input_values": input_values, "attention_mask": attention_mask, "labels": labels, "lid_labels": lid_labels}


In [34]:
for batch in loader:
    print(batch.keys())
    print("input_values shape:", batch["input_values"].shape)
    break


collate_batch called with batch of size: 2
005-1 ahí ya puse
005-2 ahí ya está


AttributeError: 'list' object has no attribute 'keys'

# NEW TRY

In [11]:
import math
import numpy as np
import soundfile as sf
import librosa
import re

# Open the audio file (16kHz mono is expected for Wav2Vec2)
audio_file = "tr-005.wav"  # replace with your actual audio filename if different
audio_sf = sf.SoundFile(audio_file)
original_sr = audio_sf.samplerate  # original sample rate of the audio file
print(f"Audio sample rate: {original_sr} Hz")

# Prepare an empty list to hold each training example
dataset = []

# Loop through each utterance in your metadata
for utt_id in utterance_meta:
    # Skip if no token-level tags available for this utterance
    if utt_id not in tokenized_utterances:
        continue

    meta = utterance_meta[utt_id]           # metadata for this utterance
    tokens = tokenized_utterances[utt_id]   # list of (token, language_tag) for this utterance

    start, end, text = meta["start"], meta["end"], meta["text"]
    duration = end - start
    if duration <= 0 or not tokens:
        # Skip utterances with invalid duration or no tokens
        continue

    # **Extract audio segment for this utterance**
    start_frame = int(start * original_sr)
    num_frames = int(duration * original_sr)
    audio_sf.seek(start_frame)  # move to the start position in the audio file
    audio_segment = audio_sf.read(frames=num_frames, dtype='float32')  # read the segment

    # If stereo audio, convert to mono by averaging channels
    if audio_segment.ndim > 1:
        audio_segment = audio_segment.mean(axis=1)

    # Resample audio to 16kHz (target sample rate for Wav2Vec2)
    target_sr = 16000
    if original_sr != target_sr:
        audio_segment = librosa.resample(audio_segment, orig_sr=original_sr, target_sr=target_sr)

    # Estimate number of 20ms frames for Wav2Vec2 encoder (320 samples @ 16kHz per frame)
    frame_count = math.ceil(len(audio_segment) / 320)
    M = len(tokens)
    if M == 0 or frame_count == 0:
        # Skip if no tokens or no frames (should not usually happen due to earlier checks)
        continue

    # **Generate frame-level language ID labels (`lid_labels`)**
    frame_labels = []
    for f in range(frame_count):
        # Determine which token corresponds to frame f
        token_index = min(int(f * M / frame_count), M - 1)
        _, lang_tag = tokens[token_index]   # second element of tuple is language tag (e.g., 's' or 'g')
        if lang_tag == 's':
            frame_labels.append(0)      # label 0 for Spanish
        elif lang_tag == 'g':
            frame_labels.append(1)      # label 1 for Guarani
        else:
            frame_labels.append(-100)   # use -100 to ignore frames not labeled (if any)
    frame_labels = np.array(frame_labels, dtype=np.int64)

    # **Clean transcript text for ASR**
    # Remove punctuation and make lowercase
    clean_text = re.sub(r'[.,?!¿¡()]', '', text).strip().lower()

    # **Append the training example to the dataset list**
    dataset.append({
        "utt_id": utt_id,
        "audio": audio_segment,   # numpy array of float32 audio samples (16kHz)
        "text": clean_text,      # cleaned transcript string
        "lid_labels": frame_labels  # numpy array of int64 language labels for each frame
    })

# Close the audio file after processing all utterances
audio_sf.close()

print(f"Prepared {len(dataset)} training examples with frame-level language labels.")
# Print an example item to verify it has all keys
if len(dataset) > 0:
    sample = dataset[0]
    print("Example item keys:", list(sample.keys()))
    print("utt_id:", sample["utt_id"])
    print("text:", sample["text"])
    print("lid_labels shape:", sample["lid_labels"].shape, "labels (first 20 frames):", sample["lid_labels"][:20])
    print("audio length (samples):", len(sample["audio"]), "data type:", sample["audio"].dtype)


Audio sample rate: 44100 Hz
Prepared 1679 training examples with frame-level language labels.
Example item keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-1
text: ahí ya puse
lid_labels shape: (34,) labels (first 20 frames): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
audio length (samples): 10720 data type: float32


In [10]:
from torch.utils.data import Dataset

# Custom Dataset class to wrap our list of dictionaries
class CSDataset(Dataset):
    def __init__(self, data_list):
        # Ensure the input is a list of dictionaries
        assert isinstance(data_list, list) and len(data_list) > 0 and isinstance(data_list[0], dict), \
            "data_list must be a list of dictionaries with at least one element"
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Return the dictionary for the given index
        return self.data[idx]

# Instantiate the custom dataset for training
train_dataset = CSDataset(dataset)

# Quick sanity check on the first item of train_dataset
print("Number of training examples:", len(train_dataset))
first_item = train_dataset[0]
print("First item keys:", list(first_item.keys()))  # should show ['utt_id', 'audio', 'text', 'lid_labels']
print("First item utt_id:", first_item["utt_id"])
print("First item text:", first_item["text"])
print("First item lid_labels length:", len(first_item["lid_labels"]))
print("First item audio tensor length:", len(first_item["audio"]))


Number of training examples: 1679
First item keys: ['utt_id', 'audio', 'text', 'lid_labels']
First item utt_id: 005-1
First item text: ahí ya puse
First item lid_labels length: 34
First item audio tensor length: 10720


In [12]:
import torch
from transformers import Trainer

class MultitaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Separate the inputs
        labels = inputs.pop("labels")
        lang_labels = inputs.pop("language_labels")
        # Forward pass (include output_hidden_states to get encoder features)
        outputs = model(**inputs, labels=labels, output_hidden_states=True)
        # CTC loss from model (this is computed when labels are provided to Wav2Vec2ForCTC)
        ctc_loss = outputs.loss
        # Get acoustic hidden representations (last_hidden_state)
        hidden_states = outputs.hidden_states[-1]  # shape: (batch, seq_length, hidden_size)
        # Compute language logits using the classifier head
        lang_logits = model.lang_classifier(hidden_states)  # shape: (batch, seq_length, num_languages)
        # Flatten the sequences for loss computation
        vocab_size = lang_logits.size(-1)
        lang_loss = torch.nn.functional.cross_entropy(
            lang_logits.view(-1, vocab_size),
            lang_labels.view(-1)
        )
        # Combine losses (with optional weight lambda for LAL)
        total_loss = ctc_loss + LAL_weight * lang_loss
        return (total_loss, outputs) if return_outputs else total_loss


In [22]:
from datasets import Dataset

# Suppose `dataset` is your list of dicts as defined earlier:
# dataset = [{'utt_id': ..., 'audio': ..., 'text': ..., 'lid_labels': ...}, ...]

hf_dataset = Dataset.from_list(dataset)   # Now it's a Hugging Face Dataset object


In [23]:
# 10% for evaluation
split_dataset = hf_dataset.train_test_split(test_size=0.1, shuffle=True, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train samples: {len(train_dataset)}, Eval samples: {len(eval_dataset)}")


Train samples: 1511, Eval samples: 168


In [24]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./csasr-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=1e-4,
    gradient_accumulation_steps=2,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",   # recommend 'epoch' for eval every epoch
    report_to="none"
)


In [31]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_batch,
    # tokenizer=processor,            # Uncomment if you have a processor/tokenizer
    # compute_metrics=compute_metrics # Uncomment if you defined a metric function
)


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_batch,
    # Add these if you have them:
    # tokenizer=processor,
    # compute_metrics=compute_metrics,
)


# Try again

In [30]:
print(type(train_dataset))
print(len(train_dataset))
print(train_dataset[0].keys())
print(train_dataset[0])


<class 'datasets.arrow_dataset.Dataset'>
1511
dict_keys(['utt_id', 'audio', 'text', 'lid_labels'])
{'utt_id': '005-11', 'audio': [-0.008474010042846203, -0.014264961704611778, -0.0124383345246315, -0.011829998344182968, -0.00839010626077652, -0.006118712015450001, -0.00494669284671545, -0.0052672624588012695, -0.0052935234270989895, -0.005590085871517658, -0.004436770919710398, -0.002668719971552491, 0.0006316162180155516, 0.0025421669706702232, 0.0041010514833033085, 0.004934321623295546, 0.005502897314727306, 0.0061550443060696125, 0.00509103387594223, 0.003644073149189353, 0.0019949565175920725, 0.002375675132498145, 0.0039498102851212025, 0.005525513552129269, 0.0061231087893247604, 0.005952916108071804, 0.0041860854253172874, 4.7706475015729666e-05, -0.003949910402297974, -0.007194849196821451, -0.007314949296414852, -0.006406118627637625, -0.0062257652170956135, -0.007137837819755077, -0.007099613547325134, -0.005664310418069363, -0.0036308474373072386, -0.00275735417380929, -0.0

In [32]:
print("Model type:", type(model))
print(model)  # You can comment out if the model printout is huge
print("Training Args:", training_args)
print("train_dataset type:", type(train_dataset))
print("Train dataset length:", len(train_dataset))
print("First item keys:", train_dataset[0].keys())
print("First item sample:", train_dataset[0])
print("eval_dataset type:", type(eval_dataset))
print("Eval dataset length:", len(eval_dataset))
print("First eval item keys:", eval_dataset[0].keys())
print("First eval item sample:", eval_dataset[0])
print("collate_batch function:", collate_batch)


Model type: <class '__main__.Wav2Vec2ForCSASR'>
Wav2Vec2ForCSASR(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,),

In [34]:
def collate_batch(batch):
    print(f"\ncollate_batch called with batch of size: {len(batch)}")
    for i, item in enumerate(batch):
        print(f"Item {i} keys: {list(item.keys())}")
        print(f"utt_id: {item.get('utt_id')}")
        print(f"text: {item.get('text', '<MISSING>')}")
    max_audio_len = max(len(item["audio"]) for item in batch)
    max_frame_count = math.ceil(max_audio_len / 320)
    max_label_len = max(len(item["text"]) for item in batch)
    input_values_list, attention_mask_list, labels_list, lid_labels_list = [], [], [], []
    for item in batch:
        # Audio: convert list to np.array if needed
        audio = item["audio"]
        if isinstance(audio, list):
            audio = np.array(audio, dtype=np.float32)
        else:
            audio = audio.astype('float32')
        aud_tensor = torch.from_numpy(audio)
        if len(aud_tensor) < max_audio_len:
            pad_len = max_audio_len - len(aud_tensor)
            aud_tensor = torch.nn.functional.pad(aud_tensor, (0, pad_len))
        input_values_list.append(aud_tensor)
        attn_mask = torch.ones(len(audio), dtype=torch.long)
        if len(attn_mask) < max_audio_len:
            pad_len = max_audio_len - len(attn_mask)
            attn_mask = torch.nn.functional.pad(attn_mask, (0, pad_len), value=0)
        attention_mask_list.append(attn_mask)
        # Transcription labels
        text = item["text"]
        char_ids = [vocab_list.index(ch) for ch in text if ch in vocab_list]
        label_ids = torch.tensor(char_ids, dtype=torch.long)
        if len(label_ids) < max_label_len:
            pad_size = max_label_len - len(label_ids)
            label_ids = torch.nn.functional.pad(label_ids, (0, pad_size), value=-100)
        labels_list.append(label_ids)
        # LID labels
        lid_labels = item["lid_labels"]
        if isinstance(lid_labels, list):
            lid_labels = torch.tensor(lid_labels, dtype=torch.long)
        else:
            lid_labels = torch.as_tensor(lid_labels, dtype=torch.long)
        frame_count = math.ceil(len(audio) / 320)
        if frame_count < max_frame_count:
            pad_frames = max_frame_count - frame_count
            lid_labels = torch.nn.functional.pad(lid_labels, (0, pad_frames), value=-100)
        lid_labels_list.append(lid_labels)
    input_values = torch.stack(input_values_list)
    attention_mask = torch.stack(attention_mask_list)
    labels = torch.stack(labels_list)
    lid_labels = torch.stack(lid_labels_list)
    print("Batch loaded!")  # Debug print
    return {
        "input_values": input_values,
        "attention_mask": attention_mask,
        "labels": labels,
        "lid_labels": lid_labels
    }


In [35]:
# Sanity check: Try loading a batch with DataLoader and collate_batch directly
from torch.utils.data import DataLoader

sample_loader = DataLoader(train_dataset, batch_size=4, collate_fn=collate_batch)
for batch in sample_loader:
    print(batch.keys())
    print({k: v.shape for k, v in batch.items() if hasattr(v, 'shape')})
    break



collate_batch called with batch of size: 4
Item 0 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-11
text: me vendo así comestibles así
Item 1 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-1520
text: no me retaron ellos y yo me enojé por eso
Item 2 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-993
text: pero ndoka'úi ajépa
Item 3 keys: ['utt_id', 'audio', 'text', 'lid_labels']
utt_id: 005-753
text: diez mil guaraníes un cajón de banana
Batch loaded!
dict_keys(['input_values', 'attention_mask', 'labels', 'lid_labels'])
{'input_values': torch.Size([4, 75360]), 'attention_mask': torch.Size([4, 75360]), 'labels': torch.Size([4, 41]), 'lid_labels': torch.Size([4, 236])}


In [39]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,        # <-- Provide your eval split here
    data_collator=collate_batch
)


In [42]:
training_args = TrainingArguments(
    output_dir="./csasr-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=1e-4,
    gradient_accumulation_steps=2,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",      # <--- Set to "no"
    report_to="none"
)


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

# ====== Hyperparameters ======
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GRAD_ACCUM_STEPS = 2

# ====== Dataset & DataLoader ======
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)

# ====== Model & Optimizer ======
model = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# ====== Training Loop ======
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    num_batches = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for batch_idx, batch in enumerate(pbar):
        input_values = batch["input_values"].to(DEVICE)         # (B, T)
        attention_mask = batch["attention_mask"].to(DEVICE)     # (B, T)
        labels = batch["labels"].to(DEVICE)                     # (B, S)
        lid_labels = batch["lid_labels"].to(DEVICE)             # (B, F)

        # ====== Forward Pass ======
        # Let the model handle all loss computation, including CTC input_lengths
        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask,
            labels=labels,
            lid_labels=lid_labels,
        )
        # (This assumes your model's forward() returns .logits and .loss)
        logits = outputs.logits                    # (B, F, vocab)
        loss = outputs.loss

        # ====== Backward/Update ======
        loss.backward()
        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix(loss=f"{running_loss/num_batches:.4f}")

    print(f"Epoch {epoch+1} average loss: {running_loss/num_batches:.4f}")

# ====== Save Model ======
torch.save(model.state_dict(), "csasr_wav2vec2_lal_finetuned.pth")
print("Model saved!")


NameError: name 'train_dataset' is not defined

In [43]:
trainer.train()

AttributeError: `AcceleratorState` object has no attribute `distributed_type`. This happens if `AcceleratorState._reset_state()` was called and an `Accelerator` or `PartialState` was not reinitialized.

In [41]:
training_args = TrainingArguments(
    output_dir="path/to/output",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=3e-4,
    fp16=True,
    label_names=["labels", "language_labels"]  # include our custom label fields
)
trainer = MultitaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor,  # use Wav2Vec2Processor (feature extractor + tokenizer)
    compute_metrics=compute_metrics_function  # if you have a WER/CER metric function
)


NameError: name 'eval_dataset' is not defined

In [13]:
from transformers import Wav2Vec2Model, Wav2Vec2Config
import torch
import torch.nn as nn
from transformers.modeling_outputs import CausalLMOutput

base_model_name = "facebook/wav2vec2-xls-r-300m"  # Multilingual is safest!
config = Wav2Vec2Config.from_pretrained(base_model_name)
config.vocab_size = vocab_size
config.pad_token_id = 0
config.ctc_loss_reduction = "mean"
config.ctc_zero_infinity = True

class Wav2Vec2ForCSASR(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(base_model_name, config=cfg)
        self.ctc_head = nn.Linear(cfg.hidden_size, cfg.vocab_size)
        self.lid_head = nn.Linear(cfg.hidden_size, 2)
        nn.init.normal_(self.ctc_head.weight, mean=0.0, std=0.1)
        nn.init.normal_(self.lid_head.weight, mean=0.0, std=0.1)
        nn.init.zeros_(self.ctc_head.bias)
        nn.init.zeros_(self.lid_head.bias)
        self.ctc_loss_fn = nn.CTCLoss(blank=1, zero_infinity=True)
    def forward(self, input_values, attention_mask=None, labels=None, lid_labels=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        logits = self.ctc_head(hidden_states)
        lid_logits = self.lid_head(hidden_states)
        log_probs = nn.functional.log_softmax(logits, dim=-1)
        loss = None
        if labels is not None:
            log_probs_t = log_probs.permute(1, 0, 2)
            batch_size, max_frames, _ = logits.shape
            input_lengths = attention_mask.sum(dim=1) if attention_mask is not None else torch.full((batch_size,), max_frames, dtype=torch.long)
            target_list, target_lengths = [], []
            for seq in labels:
                mask = seq != -100
                target_list.append(seq[mask])
                target_lengths.append(mask.sum().item())
            target_lengths = torch.tensor(target_lengths, dtype=torch.long)
            targets_concat = torch.cat(target_list) if target_list else torch.tensor([], dtype=torch.long)
            ctc_loss = self.ctc_loss_fn(log_probs_t, targets_concat, input_lengths, target_lengths)
            lid_loss = 0.0
            if lid_labels is not None:
                B, T, _ = lid_logits.shape
                lid_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
                lid_loss = lid_loss_fn(lid_logits.view(B * T, 2), lid_labels.view(-1))
            loss = ctc_loss + lid_loss
        return CausalLMOutput(loss=loss, logits=logits)

model = Wav2Vec2ForCSASR(config)
print("Model initialized.")


Model initialized.


In [15]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 37.3 MB/s eta 0:00:00


In [16]:
from jiwer import wer

def compute_wer(preds, targets):
    wers = []
    for p, t in zip(preds, targets):
        wers.append(wer(t, p))
    return sum(wers) / len(wers)

def decode_predictions(logits, vocab_list):
    pred_ids = logits.argmax(-1).cpu().numpy()
    pred_texts = []
    for ids in pred_ids:
        chars = [vocab_list[i] for i in ids if i not in [0, 1]] # remove <pad>, <blank>
        pred_texts.append(''.join(chars).replace('<pad>','').replace('<blank>','').strip())
    return pred_texts

# In your main training loop, after each epoch:
model.eval()
val_losses, preds, trues = [], [], []
with torch.no_grad():
    for batch in eval_loader:
        outputs = model(
            input_values=batch["input_values"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
            labels=batch["labels"].to(DEVICE),
            lid_labels=batch["lid_labels"].to(DEVICE),
        )
        val_losses.append(outputs.loss.item())
        pred_text = decode_predictions(outputs.logits, vocab_list)
        true_text = ["".join([vocab_list[i] for i in row if i >= 2]) for row in batch["labels"].cpu().numpy()]
        preds.extend(pred_text)
        trues.extend(true_text)
val_loss = sum(val_losses) / len(val_losses)
val_wer = compute_wer(preds, trues)
print(f"Validation loss: {val_loss:.4f}  WER: {val_wer:.4f}")


NameError: name 'eval_loader' is not defined